In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import os
import pandas as pd


FROZEN_DIR = "/content/drive/MyDrive/frozen_dataset"
FEATURE_DIR = "/content/drive/MyDrive/feature_engineering_v1"
BASELINE_METRICS_PATH = "/content/drive/MyDrive/day 15/baseline_v1/baseline_metrics.csv"
ENSEMBLE_METRICS_PATH = "/content/drive/MyDrive/day16/ensemble_v1/train_test_metrics.csv"
DAY17_METRICS_PATH = "/content/drive/MyDrive/day17/ann_v1/day17_metrics.csv"

TRAIN_PATH = f"{FROZEN_DIR}/dataset_v1_train.csv"
TEST_PATH = f"{FROZEN_DIR}/dataset_v1_test.csv"
ENGINEERED_PATH = f"{FEATURE_DIR}/engineered_dataset.csv"
FULL_PATH = f"{FROZEN_DIR}/dataset_v1_full.csv"


TARGET_COLUMN = "soc_percent"
RAW_FEATURES = ["voltage_v", "current_a", "cell_temp_c", "power_w", "ambient_temp_c"]
CHARGE_TARGET = "remaining_charge_time_s"
CHARGE_FEATURES = ["voltage_v", "current_a", "cell_temp_c", "ambient_temp_c"]


for p in [TRAIN_PATH, TEST_PATH]:
    assert os.path.exists(p), f"Couldn't find {p} — check the Day 12 copy step."

df_train_raw = pd.read_csv(TRAIN_PATH)
df_test_raw = pd.read_csv(TEST_PATH)
print(f"Frozen split loaded — train: {df_train_raw.shape}, test (untouched until the end): {df_test_raw.shape}")


HAVE_ENGINEERED = os.path.exists(ENGINEERED_PATH)
if HAVE_ENGINEERED:
    df_eng_full = pd.read_csv(ENGINEERED_PATH)
    key_cols = ["file_id", "timestamp"]
    eng_cols = [c for c in df_eng_full.columns if c not in key_cols + [TARGET_COLUMN]]
    train_ids = df_train_raw["file_id"].unique()
    test_ids = df_test_raw["file_id"].unique()
    df_eng_train = df_eng_full[df_eng_full["file_id"].isin(train_ids)].copy()
    df_eng_test = df_eng_full[df_eng_full["file_id"].isin(test_ids)].copy()
    print(f"Engineered features ({len(eng_cols)}): {eng_cols}")

Frozen split loaded — train: (76792, 7), test (untouched until the end): (27768, 7)
Engineered features (2): ['voltage_roll_std5', 'current_roll_std5']


In [6]:
prior = []
if os.path.exists(BASELINE_METRICS_PATH):
    prior.append(pd.read_csv(BASELINE_METRICS_PATH, index_col=0)[["MAE", "RMSE", "R2"]])

if os.path.exists(ENSEMBLE_METRICS_PATH):
    ens = pd.read_csv(ENSEMBLE_METRICS_PATH, index_col=0)
    ens = ens[ens.index.str.endswith("(test)")].copy()
    ens.index = ens.index.str.replace(" (test)", "", regex=False)
    prior.append(ens[["MAE", "RMSE", "R2"]])

if os.path.exists(DAY17_METRICS_PATH):
    prior.append(pd.read_csv(DAY17_METRICS_PATH, index_col=0)[["MAE", "RMSE", "R2"]])

assert prior, "No prior metrics files found — copy Day 15/16/17 outputs to Drive first."
all_prior = pd.concat(prior)

all_prior = all_prior[~all_prior.index.duplicated(keep='first')]


TUNABLE_KEYWORDS = ["RandomForest", "GradientBoosting", "XGBoost", "MLP"]
soc_rows = all_prior[~all_prior.index.str.contains("charging", case=False)]
tunable_soc = soc_rows[soc_rows.index.to_series().apply(
    lambda label: any(k in label for k in TUNABLE_KEYWORDS)
)]
assert not tunable_soc.empty, "No tunable (tree/ANN) SOC method found in prior metrics."


soc_shortlist_label = tunable_soc["MAE"].idxmin()
print(f"Auto-detected SOC shortlist (lowest test MAE among tunable methods): "
      f"'{soc_shortlist_label}' (MAE={tunable_soc.loc[soc_shortlist_label, 'MAE'].item():.3f})")


uses_engineered = "engineered" in soc_shortlist_label or "MLP" in soc_shortlist_label
if "RandomForest" in soc_shortlist_label:
    SOC_FAMILY = "RandomForest"
elif "GradientBoosting" in soc_shortlist_label or "XGBoost" in soc_shortlist_label:
    SOC_FAMILY = "GradientBoosting"
elif "MLP" in soc_shortlist_label:
    SOC_FAMILY = "MLP"

print(f"Family to tune: {SOC_FAMILY}  |  Feature set: {'engineered' if uses_engineered else 'raw'}")

Auto-detected SOC shortlist (lowest test MAE among tunable methods): 'RandomForest — raw' (MAE=3.284)
Family to tune: RandomForest  |  Feature set: raw


In [9]:
from sklearn.model_selection import GroupShuffleSplit, ParameterGrid
from sklearn.metrics import mean_absolute_error
from sklearn.ensemble import RandomForestRegressor


soc_features = eng_cols if uses_engineered else RAW_FEATURES
soc_train_df = df_eng_train if uses_engineered else df_train_raw

gss_val = GroupShuffleSplit(n_splits=1, test_size=0.15, random_state=42)
tr_idx, val_idx = next(gss_val.split(soc_train_df, groups=soc_train_df["file_id"]))
df_fit = soc_train_df.iloc[tr_idx]
df_val = soc_train_df.iloc[val_idx]

X_fit, y_fit = df_fit[soc_features], df_fit[TARGET_COLUMN]
X_val, y_val = df_val[soc_features], df_val[TARGET_COLUMN]
print(f"Validation carve — fit: {X_fit.shape}, val: {X_val.shape}")


experiment_log = []
print("===== SEARCH STRATEGY: Random Forest =====")
param_grid = {
    "n_estimators": [150, 300],
    "max_depth": [8, 12],
    "min_samples_leaf": [2,4],
}
print(f"Grid size: {len(list(ParameterGrid(param_grid)))} configurations\n")

for params in ParameterGrid(param_grid):
    model = RandomForestRegressor(random_state=42, n_jobs=-1, **params)
    model.fit(X_fit, y_fit)
    val_mae = mean_absolute_error(y_val, model.predict(X_val))
    experiment_log.append({**params, "val_MAE": val_mae})
    print(f"  {params} -> val_MAE={val_mae:.4f}")

experiment_log_df = pd.DataFrame(experiment_log).sort_values("val_MAE").reset_index(drop=True)
print("\n===== EXPERIMENT LOG (sorted best-first) =====")
print(experiment_log_df.to_string())

best_params = experiment_log_df.iloc[0].drop("val_MAE").to_dict()
best_params = {k: (int(v) if float(v).is_integer() else v) for k, v in best_params.items()}
print(f"\nBest validated configuration: {best_params} (val_MAE={experiment_log_df.iloc[0]['val_MAE']:.4f})")


print("\n===== FINAL TEST RESULT (test set touched for the first time, once) =====")
X_train_full = soc_train_df[soc_features]
y_train_full = soc_train_df[TARGET_COLUMN]
soc_test_df = df_eng_test if uses_engineered else df_test_raw
X_test_final = soc_test_df[soc_features]
y_test_final = soc_test_df[TARGET_COLUMN]

final_model = RandomForestRegressor(random_state=42, n_jobs=-1, **best_params)
final_model.fit(X_train_full, y_train_full)
y_pred_final = final_model.predict(X_test_final)

soc_final_mae = mean_absolute_error(y_test_final, y_pred_final)


print(f"Tuned {SOC_FAMILY} — MAE={soc_final_mae:.3f}")
print(f"Compare to untuned '{soc_shortlist_label}': "
      f"MAE={all_prior.loc[soc_shortlist_label, 'MAE']:.3f}  "
      f"({'improved' if soc_final_mae < all_prior.loc[soc_shortlist_label, 'MAE'] else 'no improvement — keep the untuned config'})")

Validation carve — fit: (62120, 5), val: (14672, 5)
===== SEARCH STRATEGY: Random Forest =====
Grid size: 8 configurations

  {'max_depth': 8, 'min_samples_leaf': 2, 'n_estimators': 150} -> val_MAE=1.2553
  {'max_depth': 8, 'min_samples_leaf': 2, 'n_estimators': 300} -> val_MAE=1.2489
  {'max_depth': 8, 'min_samples_leaf': 4, 'n_estimators': 150} -> val_MAE=1.2491
  {'max_depth': 8, 'min_samples_leaf': 4, 'n_estimators': 300} -> val_MAE=1.2431
  {'max_depth': 12, 'min_samples_leaf': 2, 'n_estimators': 150} -> val_MAE=1.5689
  {'max_depth': 12, 'min_samples_leaf': 2, 'n_estimators': 300} -> val_MAE=1.5628
  {'max_depth': 12, 'min_samples_leaf': 4, 'n_estimators': 150} -> val_MAE=1.4652
  {'max_depth': 12, 'min_samples_leaf': 4, 'n_estimators': 300} -> val_MAE=1.4672

===== EXPERIMENT LOG (sorted best-first) =====
   max_depth  min_samples_leaf  n_estimators   val_MAE
0          8                 4           300  1.243065
1          8                 2           300  1.248926
2          

In [10]:
df_full = pd.read_csv(FULL_PATH)
df_full["prog_time_td"] = pd.to_timedelta(df_full["prog_time"], errors="coerce")
df_full = df_full.sort_values(["file_id", "prog_time_td"]).reset_index(drop=True)
df_full["status_change"] = ((df_full["status"] != df_full["status"].shift(1)) |
                             (df_full["file_id"] != df_full["file_id"].shift(1)))
df_full["segment_id"] = df_full["status_change"].cumsum()
df_charge = df_full[df_full["status"] == "CHA"].copy()
seg_end = df_charge.groupby("segment_id")["prog_time_td"].transform("max")
df_charge[CHARGE_TARGET] = (seg_end - df_charge["prog_time_td"]).dt.total_seconds()
charge_train_full = df_charge[df_charge["file_id"].isin(df_train_raw["file_id"].unique())].copy()
charge_test_full = df_charge[df_charge["file_id"].isin(df_test_raw["file_id"].unique())].copy()
charge_train_full.dropna(subset=[CHARGE_TARGET], inplace=True)
charge_test_full.dropna(subset=[CHARGE_TARGET], inplace=True)


gss_c = GroupShuffleSplit(n_splits=1, test_size=0.15, random_state=42)
c_tr_idx, c_val_idx = next(gss_c.split(charge_train_full, groups=charge_train_full["file_id"]))
c_fit = charge_train_full.iloc[c_tr_idx]
c_val = charge_train_full.iloc[c_val_idx]


print("===== SEARCH STRATEGY: Random Forest (charging time) ====")

charge_grid = {"n_estimators": [100, 200, 300], "max_depth": [6, 10, 14]}
print(f"Grid size: {len(list(ParameterGrid(charge_grid)))} configurations\n")

charge_log = []
for params in ParameterGrid(charge_grid):
    model = RandomForestRegressor(random_state=42, n_jobs=-1, **params)
    model.fit(c_fit[CHARGE_FEATURES], c_fit[CHARGE_TARGET])
    val_mae = mean_absolute_error(c_val[CHARGE_TARGET], model.predict(c_val[CHARGE_FEATURES]))
    charge_log.append({**params, "val_MAE": val_mae})
    print(f"  {params} -> val_MAE={val_mae:.4f}")

charge_log_df = pd.DataFrame(charge_log).sort_values("val_MAE").reset_index(drop=True)
print("\n===== CHARGING-TIME EXPERIMENT LOG (best-first) ====")
print(charge_log_df.to_string())

charge_best_params = {k: int(v) for k, v in charge_log_df.iloc[0].drop("val_MAE").to_dict().items()}
print(f"\nBest validated configuration: {charge_best_params} (val_MAE={charge_log_df.iloc[0]['val_MAE']:.4f})")


final_charge_model = RandomForestRegressor(random_state=42, n_jobs=-1, **charge_best_params)
final_charge_model.fit(charge_train_full[CHARGE_FEATURES], charge_train_full[CHARGE_TARGET])
y_pred_charge = final_charge_model.predict(charge_test_full[CHARGE_FEATURES])
charge_mae = mean_absolute_error(charge_test_full[CHARGE_TARGET], y_pred_charge)

print(f"\nFINAL TEST RESULT — tuned RF (charging time): MAE={charge_mae:.3f}s")

===== SEARCH STRATEGY: Random Forest (charging time) ====
Grid size: 9 configurations

  {'max_depth': 6, 'n_estimators': 100} -> val_MAE=31215.5115
  {'max_depth': 6, 'n_estimators': 200} -> val_MAE=31217.7342
  {'max_depth': 6, 'n_estimators': 300} -> val_MAE=31243.1975
  {'max_depth': 10, 'n_estimators': 100} -> val_MAE=31213.7602
  {'max_depth': 10, 'n_estimators': 200} -> val_MAE=31215.8461
  {'max_depth': 10, 'n_estimators': 300} -> val_MAE=31241.2495
  {'max_depth': 14, 'n_estimators': 100} -> val_MAE=31214.1378
  {'max_depth': 14, 'n_estimators': 200} -> val_MAE=31216.2993
  {'max_depth': 14, 'n_estimators': 300} -> val_MAE=31241.5140

===== CHARGING-TIME EXPERIMENT LOG (best-first) ====
   max_depth  n_estimators       val_MAE
0         10           100  31213.760200
1         14           100  31214.137801
2          6           100  31215.511468
3         10           200  31215.846086
4         14           200  31216.299325
5          6           200  31217.734249
6       